In [5]:
## Embedding Techniques  
import os
from dotenv import load_dotenv

load_dotenv()


True

In [6]:
os.environ["OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_API_KEY")
os.environ["OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT")
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"
os.environ["OPENAI_DEPLOYMENT_NAME"] = "text-embedding-3-large"
os.environ["OPENAI_MODEL"] = "text-embedding-3-large"


In [7]:
from openai import AzureOpenAI
import os

# Initialize the Azure OpenAI client
client = AzureOpenAI(
    api_version=os.environ["OPENAI_API_VERSION"],
    azure_endpoint=os.environ["OPENAI_ENDPOINT"],
    api_key=os.environ["OPENAI_API_KEY"]
)


In [8]:
#Sample text to embed
response = client.embeddings.create(
    input=["first phrase","second phrase","third phrase"],
    model=os.getenv("OPENAI_MODEL")
)

for item in response.data:
    length = len(item.embedding)
    print(
        f"data[{item.index}]: length={length}, "
        f"[{item.embedding[0]}, {item.embedding[1]}, "
        f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
    )
print(response.usage)

AuthenticationError: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}

In [ ]:
# Convert text to embedding
from langchain_community.document_loaders import TextLoader
text_loader = TextLoader("speech.txt")
text_documents = text_loader.load()
text_documents

from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_documents = text_splitter.split_documents(text_documents)
final_documents


[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_co

In [ ]:
# vector embedding and vector storedb
from langchain_community.vectorstores import Chroma
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
	azure_deployment=os.environ["OPENAI_DEPLOYMENT_NAME"],
	openai_api_version=os.environ["OPENAI_API_VERSION"],
	azure_endpoint=os.environ["OPENAI_ENDPOINT"],
	openai_api_key=os.environ["OPENAI_API_KEY"]
)

vector_store = Chroma.from_documents(final_documents, embedding=embeddings)
vector_store


In [ ]:
# Search the vector store
query = "It will be all the easier for us to conduct ourselves"
results = vector_store.similarity_search(query)
for result in results:
    print(f"Document: {result.page_content}\nScore: {result.metadata.get('score', 'N/A')}\n")   

Document: It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early
Score: N/A

Document: It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire